In [ ]:
import timm
from pprint import pprint
model_names = timm.list_models(pretrained=True)
pprint(model_names)

In [ ]:
import timm
from pprint import pprint
model_names = timm.list_models('*vit*')
pprint(model_names)

In [2]:
from pathlib import Path
from typing import Optional
def _resolve_local_checkpoint(timm_name: str) -> Optional[str]:
	cache_root = Path.home() / ".cache" / "huggingface" / "hub"
	if not cache_root.exists():
		return None

	for model_dir in sorted(cache_root.glob(f"models--timm--{timm_name}*")):
		for checkpoint in sorted(model_dir.glob("snapshots/*/model.safetensors"), reverse=True):
			if checkpoint.is_file():
				return str(checkpoint)
	return None

In [3]:
import timm
import torch
x = torch.randn(1, 3, 256, 256)
_TIMM_NAME = "vit_huge_plus_patch16_dinov3_qkvb"
default_path = _resolve_local_checkpoint(_TIMM_NAME)
model = timm.create_model(_TIMM_NAME, pretrained=False, checkpoint_path=default_path)
features = model.forward_features(x)
print(features.shape)

/home/chengxiaozhen/miniconda3/envs/HF/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


torch.Size([1, 261, 1280])


In [4]:
model.pretrained_cfg

{'url': '',
 'hf_hub_id': 'timm/vit_huge_plus_patch16_dinov3_qkvb.lvd1689m',
 'architecture': 'vit_huge_plus_patch16_dinov3_qkvb',
 'tag': 'lvd1689m',
 'custom_load': False,
 'input_size': (3, 256, 256),
 'fixed_input_size': True,
 'interpolation': 'bicubic',
 'crop_pct': 1.0,
 'crop_mode': 'center',
 'mean': (0.485, 0.456, 0.406),
 'std': (0.229, 0.224, 0.225),
 'num_classes': 0,
 'pool_size': None,
 'first_conv': 'patch_embed.proj',
 'classifier': 'head',
 'license': 'dinov3-license'}

In [5]:
config = timm.data.resolve_data_config({}, model=model)
print(config)

{'input_size': (3, 256, 256), 'interpolation': 'bicubic', 'mean': (0.485, 0.456, 0.406), 'std': (0.229, 0.224, 0.225), 'crop_pct': 1.0, 'crop_mode': 'center'}


In [6]:
train_transform = timm.data.create_transform(**config,is_training=True)
train_transform

Compose(
    RandomResizedCropAndInterpolation(size=(256, 256), scale=(0.08, 1.0), ratio=(0.75, 1.3333), interpolation=bicubic)
    RandomHorizontalFlip(p=0.5)
    ColorJitter(brightness=(0.6, 1.4), contrast=(0.6, 1.4), saturation=(0.6, 1.4), hue=None)
    MaybeToTensor()
    Normalize(mean=tensor([0.4850, 0.4560, 0.4060]), std=tensor([0.2290, 0.2240, 0.2250]))
)

In [7]:
eval_transform = timm.data.create_transform(**config,is_training=False)
eval_transform

Compose(
    Resize(size=256, interpolation=bicubic, max_size=None, antialias=True)
    CenterCrop(size=(256, 256))
    MaybeToTensor()
    Normalize(mean=tensor([0.4850, 0.4560, 0.4060]), std=tensor([0.2290, 0.2240, 0.2250]))
)